#### Phase 3 - Blind Spot Analyst

This notebook bridges:
- **M1 attacks** (`strategy_results.csv`)
- **M2 fragility summary** (`min_change_summary.csv`)

It selects the most fragile samples, replays attack stages, computes **Grad-CAM** (`ResNet18.layer4`), and writes:
- Grad-CAM figures to `outputs/m3_gradcam/`
- Blind spot report to `outputs/blindspot_analysis.md`


In [1]:
import os
import re
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')


In [2]:
# Paths and config
PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'Phase 3' else Path.cwd().resolve()
PHASE3_DIR = PROJECT_ROOT / 'Phase 3'
OUTPUTS_DIR = PHASE3_DIR / 'outputs'
DATASET_TEST_DIR = PROJECT_ROOT / 'dataset' / 'test'

STRATEGY_CSV = OUTPUTS_DIR / 'strategy_results.csv'
MIN_CHANGE_CSV = OUTPUTS_DIR / 'min_change_summary.csv'
CHECKPOINT_PATH = PROJECT_ROOT / 'Phase1' / 'cifake_resnet18_final.pt'

M3_FIG_DIR = OUTPUTS_DIR / 'm3_gradcam'
M3_FIG_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUTPUTS_DIR / 'blindspot_analysis.md'

TOP_K = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PHASE3_DIR:', PHASE3_DIR)
print('STRATEGY_CSV exists:', STRATEGY_CSV.exists())
print('MIN_CHANGE_CSV exists:', MIN_CHANGE_CSV.exists())
print('CHECKPOINT_PATH exists:', CHECKPOINT_PATH.exists())
print('DEVICE:', DEVICE)


PROJECT_ROOT: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon
PHASE3_DIR: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3
STRATEGY_CSV exists: True
MIN_CHANGE_CSV exists: True
CHECKPOINT_PATH exists: True
DEVICE: cpu


In [3]:
# Model setup (matches M1 Phase 3 architecture)
def build_model() -> nn.Module:
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)
    )
    return model


model = build_model()
state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

IDX_TO_LABEL = {0: 'FAKE', 1: 'REAL'}
LABEL_TO_IDX = {v: k for k, v in IDX_TO_LABEL.items()}

print('Model loaded.')


Model loaded.


In [4]:
# M1 attack functions (re-implemented)
def apply_blur(image_np: np.ndarray, radius: float) -> np.ndarray:
    pil = Image.fromarray(image_np)
    # M1 used kernel derived from radius; here we use GaussianBlur radius directly.
    return np.array(pil.filter(ImageFilter.GaussianBlur(radius=float(radius))))


def apply_frequency_mask(image_np: np.ndarray, strength: float) -> np.ndarray:
    img_float = image_np.astype(np.float32) / 255.0
    dft = np.fft.fft2(img_float, axes=(0, 1))
    dft_shift = np.fft.fftshift(dft, axes=(0, 1))

    h, w, _ = img_float.shape
    mask = np.ones((h, w, 3), np.float32)

    center_h, center_w = h // 2, w // 2
    radius = int(min(h, w) * strength)

    mask[center_h - radius:center_h + radius, center_w - radius:center_w + radius] *= 0.5

    dft_shift *= mask
    f_ishift = np.fft.ifftshift(dft_shift, axes=(0, 1))
    img_back = np.fft.ifft2(f_ishift, axes=(0, 1))
    img_back = np.abs(img_back)

    img_back = np.clip(img_back, 0, 1)
    return (img_back * 255).astype(np.uint8)


def apply_noise(image_np: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    noise = rng.normal(0, sigma * 255, image_np.shape)
    noisy = image_np + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)


def apply_brightness(image_np: np.ndarray, factor: float) -> np.ndarray:
    bright = image_np.astype(np.float32) * factor
    return np.clip(bright, 0, 255).astype(np.uint8)


def seed_from_keys(*keys: str) -> int:
    payload = '|'.join(keys).encode('utf-8')
    return int(hashlib.md5(payload).hexdigest()[:8], 16)


def replay_attack(image_np: np.ndarray, strategy: str, parameter: float, sample_id: str, true_label: str, stage: str) -> np.ndarray:
    if strategy == 'A_blur':
        return apply_blur(image_np, float(parameter))

    if strategy == 'A_blur_freq':
        hybrid = apply_blur(image_np, 0.5)
        return apply_frequency_mask(hybrid, float(parameter))

    if strategy == 'B_noise':
        rng = np.random.default_rng(seed_from_keys(sample_id, true_label, strategy, stage))
        return apply_noise(image_np, float(parameter), rng)

    if strategy == 'B_noise_bright':
        rng = np.random.default_rng(seed_from_keys(sample_id, true_label, strategy, stage))
        hybrid = apply_noise(image_np, 0.02, rng)
        return apply_brightness(hybrid, float(parameter))

    return image_np.copy()


In [5]:
# Grad-CAM helpers
class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.fwd_handle = target_layer.register_forward_hook(self._save_activations)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input_, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor: torch.Tensor, target_class: int | None = None):
        self.model.zero_grad(set_to_none=True)
        out = self.model(input_tensor)
        probs = F.softmax(out, dim=1)

        pred_idx = int(torch.argmax(probs, dim=1).item())
        conf = float(probs[0, pred_idx].item())
        class_idx = pred_idx if target_class is None else int(target_class)

        score = out[:, class_idx]
        score.backward(retain_graph=False)

        grads = self.gradients
        acts = self.activations

        weights = grads.mean(dim=(2, 3), keepdim=True)
        cam = (weights * acts).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)

        cam = cam[0, 0].cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam, pred_idx, conf

    def close(self):
        self.fwd_handle.remove()
        self.bwd_handle.remove()


def to_tensor_for_model(image_np: np.ndarray) -> torch.Tensor:
    pil = Image.fromarray(image_np)
    return transform(pil).unsqueeze(0).to(DEVICE)


def overlay_heatmap(image_np: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    h, w = image_np.shape[:2]
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((w, h), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0
    heat = plt.cm.jet(cam_resized)[..., :3]
    img = image_np.astype(np.float32) / 255.0
    out = (1 - alpha) * img + alpha * heat
    return np.clip(out * 255.0, 0, 255).astype(np.uint8)


def cam_region_metrics(cam: np.ndarray) -> dict:
    h, w = cam.shape
    y0, y1 = int(0.25 * h), int(0.75 * h)
    x0, x1 = int(0.25 * w), int(0.75 * w)

    center = cam[y0:y1, x0:x1].mean()

    t = int(0.2 * h)
    l = int(0.2 * w)
    edge_mask = np.ones_like(cam, dtype=bool)
    edge_mask[t:h - t, l:w - l] = False
    edge = cam[edge_mask].mean()

    spread = cam.std()
    peak = cam.max()

    return {
        'center_focus': float(center),
        'edge_focus': float(edge),
        'spread': float(spread),
        'peak': float(peak),
    }


def classify_attention_change(base: dict, attacked: dict) -> str:
    dc = attacked['center_focus'] - base['center_focus']
    de = attacked['edge_focus'] - base['edge_focus']
    ds = attacked['spread'] - base['spread']

    if dc < -0.10 and de > 0.05:
        return 'shifted_to_background'
    if dc < -0.10 and ds < -0.03:
        return 'collapsed'
    if dc > 0.05 and de < 0.02:
        return 'stable_or_refocused'
    return 'mixed_shift'


In [6]:
# Load and integrate M1 + M2 outputs
strategy_df = pd.read_csv(STRATEGY_CSV)
min_change_df = pd.read_csv(MIN_CHANGE_CSV)

strategy_df['flipped'] = strategy_df['flipped'].astype(str).str.lower() == 'true'
min_change_df['flipped'] = min_change_df['flipped'].astype(str).str.lower() == 'true'

# Baseline confidence (from strategy='original', stage='original')
baseline_df = strategy_df[
    (strategy_df['strategy'] == 'original') & (strategy_df['stage'] == 'original')
][['sample_id', 'true_label', 'confidence']].rename(columns={'confidence': 'baseline_conf'})

attack_df = strategy_df[strategy_df['strategy'] != 'original'].copy()
attack_df = attack_df.merge(baseline_df, on=['sample_id', 'true_label'], how='left')
attack_df['confidence_drop'] = attack_df['baseline_conf'] - attack_df['confidence']

# Strategy-level fragility summary (fallback even if M2 has no flips)
agg = (attack_df
       .groupby(['sample_id', 'true_label', 'strategy'], as_index=False)
       .agg(
           max_conf_drop=('confidence_drop', 'max'),
           min_conf=('confidence', 'min'),
           baseline_conf=('baseline_conf', 'first'),
           any_flip=('flipped', 'max')
       ))

# Merge with M2 by sample+strategy; keep true_label from attack-derived table
m2_cols = ['sample_id', 'strategy', 'min_parameter', 'confidence_before', 'confidence_after', 'confidence_drop', 'flipped']
m2 = min_change_df[m2_cols].rename(columns={
    'confidence_drop': 'm2_confidence_drop',
    'flipped': 'm2_flipped'
})

candidates = agg.merge(m2, on=['sample_id', 'strategy'], how='left')

# Ranking: prioritize flips, then largest confidence drop (fallback from attack table)
candidates['rank_flip'] = candidates['any_flip'].astype(int)
candidates['rank_drop'] = candidates['max_conf_drop'].fillna(-1.0)

selected = (candidates
            .sort_values(['rank_flip', 'rank_drop'], ascending=[False, False])
            .head(TOP_K)
            .reset_index(drop=True))

print('Selected targets:')
print(selected[['sample_id', 'true_label', 'strategy', 'any_flip', 'max_conf_drop']])


Selected targets:
     sample_id true_label     strategy  any_flip  max_conf_drop
0  104 (3).jpg       FAKE  A_blur_freq     False       0.062030
1  105 (2).jpg       FAKE       A_blur     False       0.061439
2  105 (3).jpg       FAKE       A_blur     False       0.059152
3  104 (3).jpg       FAKE       A_blur     False       0.058577
4      105.jpg       FAKE       A_blur     False       0.050753


In [7]:
# Determine original / near-flip / flip rows for each selected case
def pick_stages(group: pd.DataFrame):
    # group is for one (sample_id, true_label, strategy)
    g = group.copy().sort_values('parameter')

    flipped_rows = g[g['flipped']]
    if not flipped_rows.empty:
        flip_row = flipped_rows.iloc[0]
        prior = g[g['parameter'] < flip_row['parameter']]
        if prior.empty:
            near_row = flip_row
        else:
            near_row = prior.iloc[-1]
    else:
        # Near-success fallback: lowest confidence stage
        near_row = g.loc[g['confidence'].idxmin()]
        flip_row = None

    return near_row, flip_row


cases = []
for _, row in selected.iterrows():
    sid = row['sample_id']
    tl = row['true_label']
    strat = row['strategy']

    sub = attack_df[(attack_df['sample_id'] == sid) & (attack_df['true_label'] == tl) & (attack_df['strategy'] == strat)]
    if sub.empty:
        continue

    near_row, flip_row = pick_stages(sub)

    base = baseline_df[(baseline_df['sample_id'] == sid) & (baseline_df['true_label'] == tl)]
    if base.empty:
        continue

    base_conf = float(base.iloc[0]['baseline_conf'])

    cases.append({
        'sample_id': sid,
        'true_label': tl,
        'strategy': strat,
        'baseline_conf': base_conf,
        'near_stage': near_row['stage'],
        'near_param': float(near_row['parameter']),
        'near_conf': float(near_row['confidence']),
        'flip_stage': None if flip_row is None else flip_row['stage'],
        'flip_param': None if flip_row is None else float(flip_row['parameter']),
        'flip_conf': None if flip_row is None else float(flip_row['confidence']),
        'flipped': flip_row is not None,
    })

cases_df = pd.DataFrame(cases)
print(cases_df)


     sample_id true_label     strategy  baseline_conf        near_stage  \
0  104 (3).jpg       FAKE  A_blur_freq       0.872472  blur0.5_freq0.15   
1  105 (2).jpg       FAKE       A_blur       0.841325          blur_0.8   
2  105 (3).jpg       FAKE       A_blur       0.867454          blur_0.8   
3  104 (3).jpg       FAKE       A_blur       0.872472          blur_0.8   
4      105.jpg       FAKE       A_blur       0.846957          blur_0.8   

   near_param  near_conf flip_stage flip_param flip_conf  flipped  
0        0.15   0.810442       None       None      None    False  
1        0.80   0.779886       None       None      None    False  
2        0.80   0.808302       None       None      None    False  
3        0.80   0.813895       None       None      None    False  
4        0.80   0.796204       None       None      None    False  


In [8]:
# Run Grad-CAM analysis and save per-sample figures
gradcam = GradCAM(model, model.layer4)
analysis_rows = []

for _, case in cases_df.iterrows():
    sid = case['sample_id']
    tl = case['true_label']
    strat = case['strategy']

    img_path = DATASET_TEST_DIR / tl / sid
    if not img_path.exists():
        print('Missing image, skipping:', img_path)
        continue

    orig_np = np.array(Image.open(img_path).convert('RGB'))

    near_np = replay_attack(
        orig_np,
        strategy=strat,
        parameter=case['near_param'],
        sample_id=sid,
        true_label=tl,
        stage=str(case['near_stage'])
    )

    if case['flipped']:
        flip_np = replay_attack(
            orig_np,
            strategy=strat,
            parameter=case['flip_param'],
            sample_id=sid,
            true_label=tl,
            stage=str(case['flip_stage'])
        )
    else:
        flip_np = near_np.copy()

    x_orig = to_tensor_for_model(orig_np)
    x_near = to_tensor_for_model(near_np)
    x_flip = to_tensor_for_model(flip_np)

    cam_o, pred_o, conf_o = gradcam.generate(x_orig)
    cam_n, pred_n, conf_n = gradcam.generate(x_near)
    cam_f, pred_f, conf_f = gradcam.generate(x_flip)

    ov_o = overlay_heatmap(orig_np, cam_o)
    ov_n = overlay_heatmap(near_np, cam_n)
    ov_f = overlay_heatmap(flip_np, cam_f)

    met_o = cam_region_metrics(cam_o)
    met_n = cam_region_metrics(cam_n)
    met_f = cam_region_metrics(cam_f)

    change_near = classify_attention_change(met_o, met_n)
    change_flip = classify_attention_change(met_o, met_f)

    fig, axes = plt.subplots(3, 2, figsize=(10, 12))

    axes[0, 0].imshow(orig_np)
    axes[0, 0].set_title(f'Original ({IDX_TO_LABEL.get(pred_o, pred_o)} {conf_o:.3f})')
    axes[0, 0].axis('off')
    axes[0, 1].imshow(ov_o)
    axes[0, 1].set_title('Grad-CAM Original')
    axes[0, 1].axis('off')

    axes[1, 0].imshow(near_np)
    axes[1, 0].set_title(f'Near-Flip: {case["near_stage"]} ({IDX_TO_LABEL.get(pred_n, pred_n)} {conf_n:.3f})')
    axes[1, 0].axis('off')
    axes[1, 1].imshow(ov_n)
    axes[1, 1].set_title(f'Grad-CAM Near ({change_near})')
    axes[1, 1].axis('off')

    axes[2, 0].imshow(flip_np)
    flip_title = case['flip_stage'] if case['flipped'] else f"fallback={case['near_stage']}"
    axes[2, 0].set_title(f'Flipped: {flip_title} ({IDX_TO_LABEL.get(pred_f, pred_f)} {conf_f:.3f})')
    axes[2, 0].axis('off')
    axes[2, 1].imshow(ov_f)
    axes[2, 1].set_title(f'Grad-CAM Flipped ({change_flip})')
    axes[2, 1].axis('off')

    fig.suptitle(f'{sid} | {tl} | {strat}', fontsize=12)
    fig.tight_layout()

    safe_name = re.sub(r'[^a-zA-Z0-9_.-]+', '_', f'{tl}_{sid}_{strat}')
    fig_path = M3_FIG_DIR / f'{safe_name}.png'
    fig.savefig(fig_path, dpi=160)
    plt.close(fig)

    analysis_rows.append({
        'sample_id': sid,
        'true_label': tl,
        'strategy': strat,
        'image_path': str(img_path),
        'figure_path': str(fig_path),
        'flipped_found': bool(case['flipped']),
        'near_stage': case['near_stage'],
        'flip_stage': case['flip_stage'] if case['flipped'] else None,
        'orig_pred': IDX_TO_LABEL.get(pred_o, pred_o),
        'near_pred': IDX_TO_LABEL.get(pred_n, pred_n),
        'flip_pred': IDX_TO_LABEL.get(pred_f, pred_f),
        'orig_conf': conf_o,
        'near_conf': conf_n,
        'flip_conf': conf_f,
        'change_near': change_near,
        'change_flip': change_flip,
        'orig_center_focus': met_o['center_focus'],
        'near_center_focus': met_n['center_focus'],
        'flip_center_focus': met_f['center_focus'],
        'orig_edge_focus': met_o['edge_focus'],
        'near_edge_focus': met_n['edge_focus'],
        'flip_edge_focus': met_f['edge_focus'],
    })

gradcam.close()

analysis_df = pd.DataFrame(analysis_rows)
analysis_csv_path = OUTPUTS_DIR / 'm3_blindspot_cases.csv'
analysis_df.to_csv(analysis_csv_path, index=False)

print('Saved figures to:', M3_FIG_DIR)
print('Saved case metrics:', analysis_csv_path)
analysis_df


Missing image, skipping: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/dataset/test/FAKE/104 (3).jpg
Missing image, skipping: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/dataset/test/FAKE/105 (2).jpg
Missing image, skipping: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/dataset/test/FAKE/105 (3).jpg
Missing image, skipping: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/dataset/test/FAKE/104 (3).jpg
Missing image, skipping: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/dataset/test/FAKE/105.jpg
Saved figures to: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m3_gradcam
Saved case metrics: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m3_blindspot_cases.csv


""


In [9]:
# Correlate attack type with attention shifts and dependency categories

def infer_dependency_categories(row: pd.Series) -> list[str]:
    categories = []

    dc = row['flip_center_focus'] - row['orig_center_focus']
    de = row['flip_edge_focus'] - row['orig_edge_focus']
    strategy = str(row['strategy'])

    # Attack-specific hypotheses based on observed shift patterns.
    if strategy in {'A_blur', 'A_blur_freq'}:
        if dc < -0.05:
            categories.append('texture patterns')
            categories.append('high-frequency artifacts')
        if de > 0.02:
            categories.append('background cues')

    if strategy in {'B_noise', 'B_noise_bright'}:
        if dc < -0.05:
            categories.append('high-frequency artifacts')
        if 'bright' in strategy.lower() and (de > 0.01 or row['change_flip'] in {'mixed_shift', 'shifted_to_background'}):
            categories.append('color distribution')
            categories.append('contrast-heavy edges')

    # Generic fallbacks from CAM behavior.
    if row['change_flip'] == 'shifted_to_background':
        categories.append('background cues')
    if row['change_flip'] == 'collapsed':
        categories.append('weakly localized evidence')

    # Keep order but unique.
    seen = set()
    out = []
    for c in categories:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out if out else ['mixed/unclear dependency']


if not analysis_df.empty:
    analysis_df['dependency_categories'] = analysis_df.apply(infer_dependency_categories, axis=1)
    analysis_df['dependency_text'] = analysis_df['dependency_categories'].apply(lambda xs: ', '.join(xs))

    # Attack-family rollups for direct correlation statements.
    def family_name(strategy: str) -> str:
        if strategy in {'A_blur', 'A_blur_freq'}:
            return 'Strategy A (Spatial + Frequency)'
        if strategy in {'B_noise', 'B_noise_bright'}:
            return 'Strategy B (Noise + Photometric)'
        return 'Other'

    analysis_df['attack_family'] = analysis_df['strategy'].map(family_name)

    attack_corr = (analysis_df
        .groupby('attack_family', as_index=False)
        .agg(
            n_cases=('sample_id', 'count'),
            mean_center_delta=('flip_center_focus', lambda s: float((s - analysis_df.loc[s.index, 'orig_center_focus']).mean())),
            mean_edge_delta=('flip_edge_focus', lambda s: float((s - analysis_df.loc[s.index, 'orig_edge_focus']).mean())),
            shifted_bg=('change_flip', lambda s: int((s == 'shifted_to_background').sum())),
            collapsed=('change_flip', lambda s: int((s == 'collapsed').sum())),
            mixed=('change_flip', lambda s: int((s == 'mixed_shift').sum())),
            stable=('change_flip', lambda s: int((s == 'stable_or_refocused').sum())),
        )
    )
else:
    attack_corr = pd.DataFrame(columns=['attack_family', 'n_cases', 'mean_center_delta', 'mean_edge_delta', 'shifted_bg', 'collapsed', 'mixed', 'stable'])

# Persist enriched case table
analysis_csv_path = OUTPUTS_DIR / 'm3_blindspot_cases.csv'
analysis_df.to_csv(analysis_csv_path, index=False)
attack_corr_path = OUTPUTS_DIR / 'm3_attack_correlation.csv'
attack_corr.to_csv(attack_corr_path, index=False)

print('Saved enriched cases:', analysis_csv_path)
print('Saved attack correlation:', attack_corr_path)
attack_corr


Saved enriched cases: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m3_blindspot_cases.csv
Saved attack correlation: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/m3_attack_correlation.csv


,attack_family,n_cases,mean_center_delta,mean_edge_delta,shifted_bg,collapsed,mixed,stable


In [10]:
# Build blindspot_analysis.md (role-aligned for Member 3)
if analysis_df.empty:
    report = [
        '# Blind Spot Analysis (Member 3)',
        '',
        'No cases were analyzed. Verify CSV/model/image paths and rerun the notebook.'
    ]
else:
    collapse_count = int((analysis_df['change_flip'] == 'collapsed').sum())
    shift_count = int((analysis_df['change_flip'] == 'shifted_to_background').sum())
    mixed_count = int((analysis_df['change_flip'] == 'mixed_shift').sum())
    stable_count = int((analysis_df['change_flip'] == 'stable_or_refocused').sum())

    avg_center_drop = float((analysis_df['flip_center_focus'] - analysis_df['orig_center_focus']).mean())
    avg_edge_gain = float((analysis_df['flip_edge_focus'] - analysis_df['orig_edge_focus']).mean())

    any_real_flip = bool(analysis_df['flipped_found'].any())

    report = []

    def _rel(p):
        try:
            return Path(p).resolve().relative_to(PROJECT_ROOT).as_posix()
        except Exception:
            return str(p)

    report.append('# Blind Spot Analysis (Member 3)')
    report.append('')
    report.append('## Inputs')
    report.append(f'- `strategy_results.csv`: `{_rel(STRATEGY_CSV)}`')
    report.append(f'- `min_change_summary.csv`: `{_rel(MIN_CHANGE_CSV)}`')
    report.append(f'- Checkpoint: `{_rel(CHECKPOINT_PATH)}`')
    report.append('')
    report.append('## Grad-CAM Comparison Protocol')
    report.append('- For each target pair: Original, Just-before-flip, Flipped (if available).')
    if not any_real_flip:
        report.append('- No true flips were present in M1/M2 outputs; the minimum-confidence stage was used as a near-success fallback for flip visualization.')
    report.append('')
    report.append('## Target Selection')
    report.append(f'- Top `{TOP_K}` sample-strategy pairs ranked by flip flag, then max confidence drop from M1 traces.')
    report.append('')
    report.append('## Attention Shift Summary')
    report.append(f'- `shifted_to_background`: **{shift_count}**')
    report.append(f'- `collapsed`: **{collapse_count}**')
    report.append(f'- `mixed_shift`: **{mixed_count}**')
    report.append(f'- `stable_or_refocused`: **{stable_count}**')
    report.append(f'- Mean center-focus change (flip - original): **{avg_center_drop:.4f}**')
    report.append(f'- Mean edge-focus change (flip - original): **{avg_edge_gain:.4f}**')
    report.append('')
    report.append('## Attack Correlation (How Attack Type Changes Attention)')
    for _, r in attack_corr.iterrows():
        report.append(
            f"- {r['attack_family']}: n={int(r['n_cases'])}, "
            f"center_delta={r['mean_center_delta']:.4f}, edge_delta={r['mean_edge_delta']:.4f}, "
            f"shifted_bg={int(r['shifted_bg'])}, collapsed={int(r['collapsed'])}, mixed={int(r['mixed'])}, stable={int(r['stable'])}"
        )
    report.append('')
    report.append('## Model Dependency Diagnosis')

    dep_counts = {}
    for cats in analysis_df['dependency_categories']:
        for c in cats:
            dep_counts[c] = dep_counts.get(c, 0) + 1

    for dep, cnt in sorted(dep_counts.items(), key=lambda x: (-x[1], x[0])):
        report.append(f'- {dep}: observed in **{cnt}** / {len(analysis_df)} cases')

    report.append('')
    report.append('## Case Findings')
    for _, r in analysis_df.iterrows():
        report.append(f"### {r['sample_id']} | {r['true_label']} | {r['strategy']}")
        report.append(f"- Figure: `{_rel(r['figure_path'])}`")
        report.append(f"- Original: `{r['orig_pred']}` ({r['orig_conf']:.3f})")
        report.append(f"- Just-before-flip `{r['near_stage']}` -> `{r['near_pred']}` ({r['near_conf']:.3f}) | attention: `{r['change_near']}`")
        if bool(r['flipped_found']):
            report.append(f"- Flipped `{r['flip_stage']}` -> `{r['flip_pred']}` ({r['flip_conf']:.3f}) | attention: `{r['change_flip']}`")
        else:
            report.append(f"- Flipped stage not available in source outputs; fallback stage reused for comparison | attention: `{r['change_flip']}`")
        report.append(f"- Dependency interpretation: `{r['dependency_text']}`")
        report.append('')

    report.append('## Recommendation Handoff')
    report.append('- Texture/high-frequency dependency: add blur/frequency suppression augmentation and spectral regularization.')
    report.append('- Background cue dependency: add face-focused cropping/segmentation preprocessing.')
    report.append('- Color/contrast dependency: strengthen photometric augmentation and color jitter robustness.')

REPORT_PATH.write_text('\n'.join(report), encoding='utf-8')
print('Saved report:', REPORT_PATH)
print('Preview:')
print('\n'.join(report[:40]))


Saved report: /Users/arulkevin/Desktop/psg_hackathon/Feb-2026-Hackathon/Phase 3/outputs/blindspot_analysis.md
Preview:
# Blind Spot Analysis (Member 3)

No cases were analyzed. Verify CSV/model/image paths and rerun the notebook.
